# Задание 3. RAG
Установка необходимых библиотек:

In [1]:
!pip install -q langchain-huggingface langchain-community transformers accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 31.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 29.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 5.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests=

In [2]:
!pip install -q langchain-qdrant qdrant-client

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 9.1 MB/s eta 0:00:00:00:01


Загрузим датасет:

In [3]:
from datasets import load_dataset

dataset = load_dataset("bearberry/rus_xquadqa", split="train")
dataset

README.md:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

rus_xquadqa.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1190 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'question', 'answers', 'normalized_answers', 'context', 'metadata'],
    num_rows: 1190
})

### Подготовка корпуса

In [4]:
seen = set()
corpus = []

for row in dataset:
    for seg in row["context"]:
        chunk = seg["chunk"]
        if chunk not in seen:
            seen.add(chunk)
            corpus.append(chunk)

len(corpus), corpus[0]

(1237,
 'Защита Пэнтерс уступила всего 308 очков, заняв шестое место в лиге, а также лидировала в НФЛ по перехватам с 24 и похвасталась четырьмя попаданиями в Пробоул.')

Создадим документы из корпуса с префиксом "passage: ".

In [5]:
from langchain_core.documents import Document

corpus_docs = [
    Document(page_content=f"passage: {text}", metadata={"doc_id": i})
    for i, text in enumerate(corpus)
]

len(corpus_docs), corpus_docs[0]

(1237,
 Document(metadata={'doc_id': 0}, page_content='passage: Защита Пэнтерс уступила всего 308 очков, заняв шестое место в лиге, а также лидировала в НФЛ по перехватам с 24 и похвасталась четырьмя попаданиями в Пробоул.'))

### Векторизация в langchain

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings
import torch

embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda" if torch.cuda.is_available() else "cpu"}
)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.11G [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/418 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [7]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

qdrant_client = QdrantClient(location=":memory:")

vector_store = QdrantVectorStore.from_documents(
    documents=corpus_docs,
    embedding=embeddings,
    location=":memory:",
    collection_name="rus_xquadqa_corpus",
)

### Поиск по БД

In [8]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

Посмотрим, как работает:

In [9]:
query = dataset[0]["question"]
retriever.invoke("query: " + query)

[Document(metadata={'doc_id': 0, '_id': '64c4ed0a9c704d3fa6bffb0db0581485', '_collection_name': 'rus_xquadqa_corpus'}, page_content='passage: Защита Пэнтерс уступила всего 308 очков, заняв шестое место в лиге, а также лидировала в НФЛ по перехватам с 24 и похвасталась четырьмя попаданиями в Пробоул.'),
 Document(metadata={'doc_id': 18, '_id': 'e6dce6fa8662400fac5f4cfa9a5749a5', '_collection_name': 'rus_xquadqa_corpus'}, page_content='passage: Затем Андерсон забил гол после перехвата с 2-ярдовом тачдауном, и Мэннинг завершил пас Бенни Фаулеру для 2-очковой конверсии, давая Денверу преимущество 24–10, когда оставалось 3:08 и, по сути, отказываясь от игры.'),
 Document(metadata={'doc_id': 15, '_id': 'be1d14cedab34ba18c53e10627afa45d', '_collection_name': 'rus_xquadqa_corpus'}, page_content='passage: В следующей игре Миллер забрал мяч у Ньютона, и после того, как несколько игроков бросились к нему, он сделал длинный отскок назад и передал мяч Уарду , который вернул его на пять ярдов к 4-яр

### Загрузка LLM и реализация поиска
Будем использовать модель Qwen2.5-1.5B-Instruct.

In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto"
)

model.eval();

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [11]:
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False
)

hugging_face_pipeline = HuggingFacePipeline(pipeline=pipe)

In [12]:
import re

def generate_answer(chat):
    prompt = tokenizer.apply_chat_template(
        chat,
        tokenize=False,
        add_generation_prompt=True
    )
    output = pipe(prompt)[0]["generated_text"]
    return output

def answer_without_rag(question):
    chat = [
        {"role": "system", "content": "Отвечай на вопрос максимально кратко, не пиши дополнительные слова и знаки препинания"},
        {"role": "user", "content": question},
    ]
    return generate_answer(chat)

def answer_with_rag(question, k=5):
    relevant_docs = retriever.invoke("query: " + question)

    docs_text = "\n\n".join(
        doc.page_content.replace("passage: ", "", 1)
        for doc in relevant_docs[:k]
    )

    chat = [
        {"role": "system", "content": "Отвечай на вопрос максимально кратко, не пиши дополнительные слова и знаки препинания"},
        {
            "role": "user",
            "content": f"Контекст:\n{docs_text}\n\nВопрос: {question}"
        },
    ]
    return generate_answer(chat)

Функция нормализации текста для более справедливого сравнения:

In [13]:
import re
import string

def normalize_text(x):
    x = str(x).lower().strip()
    x = x.translate(str.maketrans("", "", string.punctuation + "«»—–…"))
    return re.sub(r"\s+", " ", x)

В качестве метрик качества будем использовать ROUGE-L (пришлось реализовать самостоятельно, потому что rouge_score не работал с русским языком) и точное совпадение.

In [14]:
import re
import string

def normalize_text(x):
    x = str(x).lower().strip()
    x = x.translate(str.maketrans("", "", string.punctuation + "«»—–…"))
    return re.sub(r"\s+", " ", x)

def rouge_l(pred, ref):
    a = normalize_text(pred).split()
    b = normalize_text(ref).split()

    if not a or not b:
        return 0.0

    dp = [[0] * (len(b) + 1) for _ in range(len(a) + 1)]

    for i in range(1, len(a) + 1):
        for j in range(1, len(b) + 1):
            if a[i - 1] == b[j - 1]:
                dp[i][j] = dp[i - 1][j - 1] + 1
            else:
                dp[i][j] = max(dp[i - 1][j], dp[i][j - 1])

    lcs = dp[-1][-1]
    p = lcs / len(a)
    r = lcs / len(b)

    if p + r == 0:
        return 0.0

    return 2 * p * r / (p + r)

Будем брать наибольшее значение каждой из метрик, сравнивая ответ модели с эталонными ответами из answers и normalized_answers.

In [ ]:
results = []

for row in dataset.select(range(10)):
    refs = list(set(row["answers"]) | set(row["normalized_answers"]))

    pred_no_rag = str(answer_without_rag(row["question"]))
    pred_rag = str(answer_with_rag(row["question"]))

    p1 = normalize_text(pred_no_rag)
    p2 = normalize_text(pred_rag)
    
    em_no_rag = max((p1 == normalize_text(ref) for ref in refs), default=0)
    em_rag = max((p2 == normalize_text(ref) for ref in refs), default=0)
    
    rougeL_no_rag = max((rouge_l(pred_no_rag, ref) for ref in refs), default=0)
    rougeL_rag = max((rouge_l(pred_rag, ref) for ref in refs), default=0)

    results.append({
        "question": row["question"],
        "answers": refs,
        "pred_no_rag": pred_no_rag,
        "pred_rag": pred_rag,
        "em_no_rag": em_no_rag,
        "em_rag": em_rag,
        "rougeL_no_rag": rougeL_no_rag,
        "rougeL_rag": rougeL_rag,
    })

Посмотрим, что получилось:

In [20]:
import pandas as pd

results_df = pd.DataFrame(results)
results_df


,question,answers,pred_no_rag,pred_rag,em_no_rag,em_rag,rougeL_no_rag,rougeL_rag
0,Сколько очков уступила защита Пэнтерс?,[308],6,308,False,True,0.0,1.000000
1,Сколько мешков за карьеру было у Джареда Аллена?,[136],40,136,False,True,0.0,1.000000
2,Сколько блокировок записал на свой счет Люк Ки...,[118],1064 блокировки,118,False,True,0.0,1.000000
3,Сколько мячей перехватил Джош Норман?,"[четыре, 4]",20,4 мяча,False,False,0.0,0.666667
4,Кто больше записал на свой счет мешков в коман...,"[кейван шорты, Кейван Шорт]",Марк Симонец,Джаред Аллен.,False,False,0.0,0.000000
5,Сколько перехватов приписывают защите Пэнтерс ...,[24],7,24,False,True,0.0,1.000000
6,Кто был лидером Пэнтерс по мешкам?,"[кейван шорты, Кейван Шорт]",Джош Энтони,Джаред Аллен,False,False,0.0,0.000000
7,Сколько игроков защиты Пэнтерс было выбрано дл...,"[четыре, 4]",12,Три.,False,False,0.0,0.000000
8,Сколько вынужденных потерь мяча имел Томас Дэвис?,"[четыре, 4]",50 мячей,4,False,True,0.0,1.000000
9,У какого игрока было больше всего перехватов в...,"[курт колеман, Курт Колеман]",Джеймс Хиллариоу.,Курт Колеман.,False,True,0.0,1.000000


Посчитаем средние метрики:

In [21]:
summary_df = pd.DataFrame([
    {
        "system": "without_rag",
        "EM": results_df["em_no_rag"].mean(),
        "ROUGE-L": results_df["rougeL_no_rag"].mean(),
    },
    {
        "system": "with_rag",
        "EM": results_df["em_rag"].mean(),
        "ROUGE-L": results_df["rougeL_rag"].mean(),
    }
])

summary_df


,system,EM,ROUGE-L
0,without_rag,0.0,0.000000
1,with_rag,0.6,0.666667


### Выводы
Как и ожидалось, система с RAG работает лучше, но не идеально. Возможно, качество работы можно улучшить, увеличив объем подаваемого контекста (например, 10 документов вместо 5).